# QAOA angle prediction — data generation + baseline MLP

**Task.** For a fixed 12-qubit Ising instance `J`, predict the depth-5 QAOA angles
$(\gamma_1..\gamma_5,\ \beta_1..\beta_5)$ that maximise $P(\text{ground})$ for each linear-field
vector $h$. Scored by mean $P(\text{ground})$ over `h_test`, measured with the organisers' `QAOA.py`.

**The catch:** `h_train.npy` ships with **no labels**. The optimal angles are ours to produce.
So most of the work here is *data generation*, not modelling.

Order of business:

1. **Read the data** — and find that `J` is exactly rank-1, which hands us a closed-form
   energy spectrum and a 128-element symmetry group.
2. **Fix reference points** — random angles, and the best *constant* angle vector. Constant
   submissions are banned, but that number is the honest bar for "does the model use $h$ at all?".
3. **Scout the landscape** — it is badly multi-modal, which dictates how labels must be
   generated and how angles must be encoded.
4. **Generate labels** — multi-restart optimisation over synthetic `h`, batched on the GPU.
5. **Train a trivial MLP** and score it with the organisers' simulator.

This is a **baseline**: plain MLP, MSE loss. The value is the defensible floor plus the two
diagnostics that decide the final score — the symmetry structure and the label multi-modality.

---
**Environment.** Written for a **Kaggle GPU kernel** (also runs on Colab or any CUDA box; falls
back to CPU, slowly). Self-contained: no repo clone, no extra pip installs. The only external
requirement is the three organiser files — `J.npy`, `h_train.npy`, `QAOA.py` — discovered
automatically under `/kaggle/input/` (any dataset slug) or `./data/raw/`.

## 0. Environment and data discovery

In [ ]:
import os, sys, glob, time, math, csv, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
else:
    print("  WARNING: no GPU — generation will be very slow. Shrink CFG below.")

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)


def find_file(name, extra_dirs=()):
    """Locate an organiser file across Kaggle input, Colab, or a local checkout."""
    roots = list(extra_dirs) + ["/kaggle/input", "/kaggle/working", "data/raw",
                                "../data/raw", ".", "..", "/content"]
    for root in roots:
        if not os.path.isdir(root):
            continue
        hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(
        f"Could not find {name}. Attach the competition files as a Kaggle dataset "
        f"(J.npy, h_train.npy, QAOA.py) or put them in ./data/raw/.")


J_PATH = find_file("J.npy")
H_PATH = find_file("h_train.npy")
QAOA_PATH = find_file("QAOA.py")
print(f"\nJ.npy       {J_PATH}\nh_train.npy {H_PATH}\nQAOA.py     {QAOA_PATH}")

sys.path.insert(0, os.path.dirname(os.path.abspath(QAOA_PATH)))
from QAOA import QAOA, P as P_DEPTH, N_QUBITS   # organisers' simulator = the metric of record

J = np.load(J_PATH).astype(np.float64)
h_train = np.load(H_PATH).astype(np.float64)
DIM = 2 ** N_QUBITS
print(f"\nJ {J.shape} | h_train {h_train.shape} | p={P_DEPTH} | n={N_QUBITS} | dim={DIM}")

### Run configuration

`CFG` is the only knob. The generation cell **measures throughput on a pilot batch and prints a
projected wall-clock time before committing** to the full run, so you can abort and lower these
rather than discover the cost an hour in.

`ROWS_PER_CHUNK` bounds GPU memory: the adjoint solver keeps ~6 live `(rows, 4096)` complex64
tensors, i.e. roughly `rows * 200 KiB`. 8192 rows ≈ 1.6 GiB.

In [ ]:
CFG = dict(
    n_new          = 8000,   # synthetic h vectors to label (on top of the 500 official ones)
    n_restarts     = 24,     # independent Adam ascents per instance  (see §4: this matters a lot)
    steps          = 350,    # ascent steps in the explore stage
    polish_steps   = 400,    # extra steps spent only on the winning restart
    lr             = 0.06,
    rows_per_chunk = 8192,   # GPU batch size in (instance x restart) rows
)
QUICK = False                # True -> a ~2 minute smoke run, for checking the notebook end-to-end
if QUICK:
    CFG.update(n_new=300, n_restarts=8, steps=120, polish_steps=120)

CACHE = "labels_main.npz" if not QUICK else "labels_quick.npz"
print(json.dumps(CFG, indent=2), f"\ncache file: {CACHE}")

## 1. What the data actually is

### 1.1 `h` is i.i.d. uniform

This is the licence to synthesise training data: we can draw as many extra instances as we like
from exactly the right distribution.

In [ ]:
ks = stats.kstest(h_train.ravel(), "uniform", args=(-1, 2))
print(f"h_train vs U(-1,1) :  KS={ks.statistic:.4f}  p={ks.pvalue:.3f}")
print(f"per-column std     :  {np.round(h_train.std(0), 3)}   (U(-1,1) -> 0.577)")
corr = np.corrcoef(h_train.T)
print(f"max |off-diag corr|:  {np.abs(corr - np.eye(N_QUBITS)).max():.3f}  (no cross-structure)")
print("\n=> h ~ i.i.d. U(-1,1). Synthetic instances are drawn the same way.")

### 1.2 `J` is exactly rank-1 — and that is a gift

`J` is not a generic random coupling matrix. It factorises **exactly**:

$$J = v v^{\mathsf T} - \operatorname{diag}(v^2), \qquad v_i = \cos(\pi k_i / 5.5),\quad
k=[0,3,5,2,1,4,4,1,2,5,3,0]$$

Two consequences:

**(a) Closed-form spectrum.** $E(z) = \tfrac12 (v\!\cdot\!z)^2 - \tfrac12\|v\|^2 + h\!\cdot\!z$.
The energy depends on the spin configuration through just two scalars — cheap, informative features.

**(b) A 128-element symmetry group.** $v$ takes 6 distinct values, each on exactly one pair
$(i,\,11-i)$. Swapping qubits within those pairs leaves $J$ invariant ($2^6 = 64$); the global spin
flip $h \to -h$ leaves $P(\text{ground})$ invariant (another $\times 2$). **Every instance in an orbit
has literally the same optimal angles.**

Both claims are asserted numerically below rather than taken on faith.

In [ ]:
import itertools

PAIRS = [(i, N_QUBITS - 1 - i) for i in range(N_QUBITS // 2)]   # (0,11) ... (5,6)


def rank1_vector(J):
    """Recover v with J = v v^T - diag(v^2). The zeroed diagonal rules out a plain
    eigendecomposition, so we solve algebraically: v_a^2 = J_ab J_ac / J_bc."""
    A = np.abs(J) + np.eye(len(J)) * np.abs(J).max()      # mask the (zero) diagonal
    a, b, c = np.argsort(-A.min(axis=1))[:3]              # best-conditioned triple
    va2 = J[a, b] * J[a, c] / J[b, c]
    v = J[:, a] / np.sqrt(va2)
    v[a] = np.sqrt(va2)
    return v if v[0] >= 0 else -v


def spin_table(n=N_QUBITS):
    """(2**n, n) table of +-1 spins in QAOA.py's bit order (qubit 0 = MSB)."""
    bits = np.arange(2 ** n)
    return 2.0 * ((bits[:, None] >> np.arange(n - 1, -1, -1)) & 1) - 1.0


def symmetry_permutations():
    """The 64 qubit permutations that leave J invariant."""
    perms = []
    for flips in itertools.product([False, True], repeat=len(PAIRS)):
        p = np.arange(N_QUBITS)
        for flip, (i, j) in zip(flips, PAIRS):
            if flip:
                p[i], p[j] = j, i
        perms.append(p)
    return np.array(perms)


def orbit(h):
    """All 128 symmetry images of each row of h -> (len(h)*128, 12)."""
    H = np.atleast_2d(h)[:, symmetry_permutations()].reshape(-1, N_QUBITS)
    return np.concatenate([H, -H], axis=0)


v = rank1_vector(J)
recon = np.outer(v, v); np.fill_diagonal(recon, 0.0)
PERMS = symmetry_permutations()
S = spin_table()

print(f"rank-1 reconstruction error : {np.abs(recon - J).max():.3e}")
print(f"v                           : {np.round(v, 4)}")
print(f"distinct values of v        : {len(np.unique(np.round(v, 6)))}  -> {len(PERMS)} permutations")
print(f"all permutations preserve J : {all(np.abs(J[np.ix_(p, p)] - J).max() < 1e-12 for p in PERMS)}")

# closed-form spectrum vs the reference simulator
ref = QAOA(torch.tensor(J, dtype=torch.float32), device=DEVICE)   # runs on GPU too
E_closed = 0.5 * ((S @ v) ** 2 - (v ** 2).sum())[None, :] + h_train[:4] @ S.T
E_ref = ref.energies(torch.tensor(h_train[:4], dtype=torch.float32)).cpu().numpy()
print(f"closed-form spectrum error  : {np.abs(E_closed - E_ref).max():.2e}")

# the claim that actually matters: P(ground) is constant along a symmetry orbit
H_orbit = orbit(h_train[:1])
g_probe = torch.tensor(rng.uniform(-1, 1, (1, P_DEPTH)), dtype=torch.float32)
b_probe = torch.tensor(rng.uniform(-1, 1, (1, P_DEPTH)), dtype=torch.float32)
with torch.no_grad():
    p_orb = ref.p_ground(torch.tensor(H_orbit, dtype=torch.float32),
                         g_probe.expand(len(H_orbit), -1),
                         b_probe.expand(len(H_orbit), -1)).cpu().numpy()
print(f"\nP(ground) spread over a {len(H_orbit)}-element orbit: {np.ptp(p_orb):.2e} "
      f"(at P={p_orb[0]:.5f})")
print("=> the 128 images are the same problem wearing different labels.")

**How we use it.** Two options: augment the training set 128×, or build features that are
*invariant by construction*. We take the second — strictly stronger, since the model cannot waste
capacity learning a symmetry we already know, and it holds exactly at test time. The invariance is
asserted in §6.

In [ ]:
E_all = 0.5 * ((S @ v) ** 2 - (v ** 2).sum())[None, :] + h_train @ S.T
gmin = E_all.min(axis=1, keepdims=True)
degeneracy = (E_all <= gmin + 1e-9).sum(1)
gap = np.sort(E_all, axis=1)[:, 1] - gmin[:, 0]
print(f"ground-state degeneracy : {np.unique(degeneracy)}   (unique ground state everywhere)")
print(f"spectral gap            : mean {gap.mean():.3f}, min {gap.min():.4f}")
print(f"=> P(ground) targets ONE basis state out of {DIM}; random angles give ~{1/DIM:.5f}")

## 2. A fast GPU simulator with exact gradients

Label generation needs tens of millions of gradient evaluations, so the search loop needs to be
fast. Running `QAOA.py` under autograd works, but it stores ~65 intermediate `(B, 4096)` complex
tensors per forward pass, which caps the batch size well below what the GPU could otherwise hold.

Since every QAOA gate is **unitary**, the backward pass can *undo* gates instead of replaying
stored ones (the adjoint method). We carry exactly two states — $\psi$ and its adjoint $\lambda$ —
walk the circuit backwards, and read off all 10 gradients on the way. Memory is $O(1)$ in depth,
so batches can be ~an order of magnitude larger.

The gradients, derived for this circuit ($\lambda$ back-propagated to just after phase layer $l$):

$$\frac{\partial P}{\partial \gamma_l} = -2\,\operatorname{Im}\langle\lambda|E|\psi\rangle,
\qquad
\frac{\partial P}{\partial \beta_l} = 2\,\operatorname{Im}\Big\langle\lambda\Big|\textstyle\sum_k X_k\Big|\psi\Big\rangle$$

This is only worth using if it is exactly right, so the next cell **asserts** values *and* both
gradients against `QAOA.py`. Note the division of labour throughout the notebook: this solver is
used only to **search**; every reported $P(\text{ground})$ is recomputed with the organisers' simulator.

In [ ]:
class FastQAOA:
    """Batched QAOA with adjoint (O(1)-memory) gradients. Conventions follow QAOA.py exactly."""

    def __init__(self, J, device=DEVICE, n=N_QUBITS, p=P_DEPTH):
        Js = (np.asarray(J, float) + np.asarray(J, float).T) / 2
        Js = Js.copy(); np.fill_diagonal(Js, 0.0)
        self.n, self.p, self.dim, self.device = n, p, 2 ** n, device
        self.cdtype = torch.complex64
        S = spin_table(n)
        self.S = torch.tensor(S, dtype=torch.float32, device=device)
        self.quad = torch.tensor(0.5 * np.einsum("ij,ki,kj->k", Js, S, S),
                                 dtype=torch.float32, device=device)

    def _t(self, x):
        return torch.as_tensor(x, dtype=torch.float32, device=self.device)

    def energies(self, h):
        h = self._t(h)
        if h.ndim == 1:
            h = h.unsqueeze(0)
        return self.quad.unsqueeze(0) + h @ self.S.T           # (B, dim)

    def ground_mask(self, E):
        return (E <= E.min(dim=1, keepdim=True).values + 1e-9).to(torch.float32)

    def _mix(self, psi, beta, dagger=False):
        """Apply prod_k exp(-i beta X_k), or its adjoint."""
        B, n, dim = psi.shape[0], self.n, self.dim
        s = -1.0 if dagger else 1.0
        cb = torch.complex(torch.cos(beta), torch.zeros_like(beta)).view(B, 1, 1)
        sb = torch.complex(torch.zeros_like(beta), -s * torch.sin(beta)).view(B, 1, 1)
        for k in range(n):
            vw = psi.view(B, 2 ** k, 2, 2 ** (n - 1 - k))
            a, c = vw[:, :, 0, :], vw[:, :, 1, :]
            psi = torch.stack([cb * a + sb * c, sb * a + cb * c], dim=2).reshape(B, dim)
        return psi

    def _sumX_expect(self, lam, psi):
        """<lam| sum_k X_k |psi> per row, without materialising the sum."""
        B, n = psi.shape[0], self.n
        acc = torch.zeros(B, dtype=self.cdtype, device=psi.device)
        for k in range(n):
            vw = psi.view(B, 2 ** k, 2, 2 ** (n - 1 - k))
            lw = lam.view(B, 2 ** k, 2, 2 ** (n - 1 - k))
            acc = acc + (lw[:, :, 0, :].conj() * vw[:, :, 1, :]).sum(dim=(1, 2))
            acc = acc + (lw[:, :, 1, :].conj() * vw[:, :, 0, :]).sum(dim=(1, 2))
        return acc

    @staticmethod
    def _phase(gamma_col, E, sign):
        ang = (sign * gamma_col) * E
        return torch.polar(torch.ones_like(ang), ang)

    def forward(self, E, gamma, beta):
        B = E.shape[0]
        psi = torch.full((B, self.dim), 1.0 / math.sqrt(self.dim),
                         dtype=self.cdtype, device=E.device)
        for l in range(self.p):
            psi = psi * self._phase(gamma[:, l:l+1], E, 1.0)
            psi = self._mix(psi, beta[:, l])
        return psi

    @torch.no_grad()
    def value_and_grad(self, E, mask, gamma, beta):
        psi = self.forward(E, gamma, beta)
        val = (mask * psi.abs() ** 2).sum(dim=1)
        lam = mask.to(self.cdtype) * psi

        gG, gB = torch.empty_like(gamma), torch.empty_like(beta)
        for l in range(self.p - 1, -1, -1):
            psi = self._mix(psi, beta[:, l], dagger=True)     # -> state just after phase l
            lam = self._mix(lam, beta[:, l], dagger=True)
            gB[:, l] = 2.0 * self._sumX_expect(lam, psi).imag
            gG[:, l] = -2.0 * (lam.conj() * (E * psi)).sum(dim=1).imag
            ph = self._phase(gamma[:, l:l+1], E, -1.0)
            psi, lam = psi * ph, lam * ph
        return val, gG, gB

    @torch.no_grad()
    def p_ground(self, h, gamma, beta):
        E = self.energies(h)
        psi = self.forward(E, self._t(gamma), self._t(beta))
        return (self.ground_mask(E) * psi.abs() ** 2).sum(dim=1)


@torch.no_grad()
def adam_ascend(sim, E, mask, gamma, beta, steps, lr, lr_final_frac=0.04):
    """Batched Adam *ascent* on P(ground) with cosine decay; every row is independent."""
    g, b = gamma.clone(), beta.clone()
    mg = torch.zeros_like(g); vg = torch.zeros_like(g)
    mb = torch.zeros_like(b); vb = torch.zeros_like(b)
    b1, b2, eps = 0.9, 0.999, 1e-8
    for t in range(1, steps + 1):
        cur = lr * (lr_final_frac + (1 - lr_final_frac)
                    * 0.5 * (1 + math.cos(math.pi * (t - 1) / steps)))
        _, dg, db = sim.value_and_grad(E, mask, g, b)
        mg.mul_(b1).add_(dg, alpha=1 - b1); vg.mul_(b2).addcmul_(dg, dg, value=1 - b2)
        mb.mul_(b1).add_(db, alpha=1 - b1); vb.mul_(b2).addcmul_(db, db, value=1 - b2)
        c1, c2 = 1 - b1 ** t, 1 - b2 ** t
        g += cur * (mg / c1) / ((vg / c2).sqrt() + eps)        # ascent
        b += cur * (mb / c1) / ((vb / c2).sqrt() + eps)
    val, _, _ = sim.value_and_grad(E, mask, g, b)
    return g, b, val


sim = FastQAOA(J)
print("FastQAOA ready on", DEVICE)

In [ ]:
# ---- correctness gate: values AND gradients vs the organisers' QAOA.py -------------------
_h = torch.tensor(h_train[:8], dtype=torch.float32, device=DEVICE)
_g = torch.tensor(rng.uniform(-1.5, 1.5, (8, P_DEPTH)), dtype=torch.float32, device=DEVICE)
_b = torch.tensor(rng.uniform(-1.5, 1.5, (8, P_DEPTH)), dtype=torch.float32, device=DEVICE)

_E = sim.energies(_h)
_val, _gG, _gB = sim.value_and_grad(_E, sim.ground_mask(_E), _g, _b)

_tg = _g.clone().requires_grad_(True)
_tb = _b.clone().requires_grad_(True)
_ref_val = ref.p_ground(_h, _tg, _tb)
_ref_val.sum().backward()

errs = {
    "value":      (_val - _ref_val.detach()).abs().max().item(),
    "grad gamma": (_gG - _tg.grad).abs().max().item(),
    "grad beta":  (_gB - _tb.grad).abs().max().item(),
}
for k, e in errs.items():
    print(f"  {k:11s} max abs error vs QAOA.py : {e:.2e}")
assert max(errs.values()) < 2e-3, "fast simulator disagrees with the reference - STOP"
print("\nOK - the search solver reproduces the organisers' simulator exactly.")

## 3. Reference points

Three numbers to frame everything that follows.

- **Random angles** — the floor ($\approx 1/4096$).
- **Best constant angles** — one angle vector tuned for the mean $P$ over many instances.
  Submitting constants scores **0** by the rules, but a model that cannot beat this has
  learned nothing about $h$.
- **Per-instance oracle** (§5) — what our own label search achieves; the ceiling for any model
  trained on those labels.

In [ ]:
def p_ground_ref(h, gamma, beta, batch=1024):
    """P(ground) under the organisers' simulator. Every headline number goes through here."""
    out = []
    with torch.no_grad():
        for i in range(0, len(h), batch):
            out.append(ref.p_ground(
                torch.tensor(h[i:i+batch], dtype=torch.float32, device=DEVICE),
                torch.tensor(np.asarray(gamma)[i:i+batch], dtype=torch.float32, device=DEVICE),
                torch.tensor(np.asarray(beta)[i:i+batch], dtype=torch.float32, device=DEVICE),
            ).cpu().numpy())
    return np.concatenate(out)


g_rand = rng.uniform(-np.pi/2, np.pi/2, (len(h_train), P_DEPTH))
b_rand = rng.uniform(-np.pi/2, np.pi/2, (len(h_train), P_DEPTH))
p_random = p_ground_ref(h_train, g_rand, b_rand)
print(f"random angles        : P = {p_random.mean():.5f}")

# --- best CONSTANT angles: ONE shared vector maximising the mean over a sample of h ---
# Restarted, because this landscape is multi-modal too (see section 4).
h_fit = torch.tensor(h_train[:256], dtype=torch.float32, device=DEVICE)
best_const, best_val = None, -1.0
for trial in range(16):
    gc = torch.tensor(rng.uniform(-np.pi/2, np.pi/2, (1, P_DEPTH)),
                      dtype=torch.float32, device=DEVICE, requires_grad=True)
    bc = torch.tensor(rng.uniform(-np.pi/2, np.pi/2, (1, P_DEPTH)),
                      dtype=torch.float32, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([gc, bc], lr=0.05)
    for _ in range(250):
        opt.zero_grad()
        loss = -ref.p_ground(h_fit, gc.expand(len(h_fit), -1),
                             bc.expand(len(h_fit), -1)).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        val = ref.p_ground(h_fit, gc.expand(len(h_fit), -1),
                           bc.expand(len(h_fit), -1)).mean().item()
    if val > best_val:
        best_val = val
        best_const = (gc.detach().cpu().numpy().copy(), bc.detach().cpu().numpy().copy())

G_CONST = np.repeat(best_const[0], len(h_train), 0)
B_CONST = np.repeat(best_const[1], len(h_train), 0)
p_const = p_ground_ref(h_train, G_CONST, B_CONST)
print(f"best constant angles : P = {p_const.mean():.5f}   <- the bar for 'the model uses h'")
print(f"   gamma = {np.round(best_const[0][0], 3)}")
print(f"   beta  = {np.round(best_const[1][0], 3)}")

## 4. Landscape reconnaissance — why this is a labelling problem

Before spending a GPU-hour generating labels, it is worth knowing what the surface looks like.
Many independent restarts on a handful of instances, compared.

In [ ]:
def init_angles(n, rng, ramp_frac=0.34):
    """Restart pool: a linear-ramp (Trotterised annealing) family plus uniform noise.

    The ramp family is a discretised slow adiabatic schedule and lands in the good basin far
    more often than uniform sampling — but on its own it is too narrow, hence the mix."""
    g = rng.uniform(-np.pi/2, np.pi/2, (n, P_DEPTH))
    b = rng.uniform(-np.pi/2, np.pi/2, (n, P_DEPTH))
    n_ramp = int(n * ramp_frac)
    if n_ramp:
        dt = rng.uniform(0.2, 1.4, (n_ramp, 1))
        l = np.arange(P_DEPTH)[None, :]
        g[:n_ramp] = (l + 1) / P_DEPTH * dt
        b[:n_ramp] = (1 - l / P_DEPTH) * dt
    return (torch.tensor(g, dtype=torch.float32, device=DEVICE),
            torch.tensor(b, dtype=torch.float32, device=DEVICE))


N_PROBE, N_RESTART = 16, 32
h_probe = np.repeat(h_train[:N_PROBE], N_RESTART, axis=0)
E_p = sim.energies(h_probe); m_p = sim.ground_mask(E_p)
g0, b0 = init_angles(N_PROBE * N_RESTART, np.random.default_rng(3))
t0 = time.time()
gp, bp, vp = adam_ascend(sim, E_p, m_p, g0, b0, steps=400, lr=0.06)
print(f"{N_PROBE*N_RESTART} rows x 400 steps in {time.time()-t0:.1f}s "
      f"({N_PROBE*N_RESTART*400/(time.time()-t0):.0f} row-steps/s)\n")

vp = vp.view(N_PROBE, N_RESTART).cpu().numpy()
gp = gp.view(N_PROBE, N_RESTART, P_DEPTH).cpu().numpy()
bp = bp.view(N_PROBE, N_RESTART, P_DEPTH).cpu().numpy()

print(f"per-instance BEST restart   : {vp.max(1).mean():.4f}")
print(f"per-instance MEDIAN restart : {np.median(vp, axis=1).mean():.4f}")
print(f"per-instance WORST restart  : {vp.min(1).mean():.4f}")
print("\nbest-of-k restarts (as a fraction of best-of-32):")
for k in [1, 2, 4, 8, 16, 32]:
    boot = np.array([[vp[i][np.random.default_rng(s).permutation(N_RESTART)[:k]].max()
                      for s in range(64)] for i in range(N_PROBE)])
    print(f"  k={k:2d}: {boot.mean():.4f}  ({boot.mean()/vp.max(1).mean()*100:.0f}%)")
print("\n=> one local optimisation is NOT a label. Restarts are mandatory, and the curve\n"
      "   is still climbing at k=32 — label quality is bought with restarts.")

In [ ]:
def wrap_pi(x):
    """Wrap into [-pi/2, pi/2) — exact, because beta is pi-periodic (see below)."""
    return (x + np.pi/2) % np.pi - np.pi/2


def canonicalize(gamma, beta):
    """Use the two exact angle symmetries to remove label ambiguity. P(ground) is unchanged.

    1. psi(-gamma, -beta) = conj psi(gamma, beta)  -> flip signs so gamma_0 >= 0.
    2. exp(-i(beta+pi)X) contributes -1 per qubit and (-1)^12 = +1 -> beta is exactly
       pi-periodic, so it can be wrapped into a half-open interval of length pi.
    """
    sign = np.where(gamma[..., :1] < 0, -1.0, 1.0)
    return gamma * sign, wrap_pi(beta * sign)


best_idx = vp.argmax(1)
G = np.stack([gp[i, best_idx[i]] for i in range(N_PROBE)])
B = np.stack([bp[i, best_idx[i]] for i in range(N_PROBE)])
G, B = canonicalize(G, B)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for i in range(N_PROBE):
    ax[0].plot(range(1, P_DEPTH+1), G[i], "o-", alpha=.6)
    ax[1].plot(range(1, P_DEPTH+1), B[i], "o-", alpha=.6)
ax[0].set_title("gamma schedule of the winning restart"); ax[0].set_xlabel("layer")
ax[1].set_title("beta schedule"); ax[1].set_xlabel("layer")
ax[1].axhline(np.pi/2, ls=":", c="k"); ax[1].axhline(-np.pi/2, ls=":", c="k")
ax[2].bar(np.arange(P_DEPTH)-0.18, G.std(0), .36, label="gamma")
ax[2].bar(np.arange(P_DEPTH)+0.18, B.std(0), .36, label="beta")
ax[2].axhline(np.pi/np.sqrt(12), ls="--", c="r", label="uniform on [-pi/2,pi/2]")
ax[2].set_title("spread of the label across instances")
ax[2].set_xticks(range(P_DEPTH)); ax[2].set_xticklabels([f"L{i+1}" for i in range(P_DEPTH)])
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"gamma std per layer: {np.round(G.std(0), 3)}")
print(f"beta  std per layer: {np.round(B.std(0), 3)}   (uniform reference: {np.pi/np.sqrt(12):.3f})")

**Two findings that shape everything downstream.**

1. **Early layers are consistent, late layers are not.** $\gamma_1,\gamma_2$ cluster tightly across
   instances while the later layers spread about as wide as a uniform draw. Good solutions for
   different $h$ sit in *different basins*, and MSE regression averages across basins — but the
   average of two good angle vectors is generally a **bad** one. **This, not model capacity, is
   what caps a regression baseline**, and §10 lists the ways out.

2. **$\beta$ piles up at $\pm\pi/2$** — a wrap artefact, not physics. $\exp(-i(\beta+\pi)X)$
   contributes $-1$ per qubit and $(-1)^{12}=+1$, so **$\beta$ is exactly $\pi$-periodic**:
   $+\pi/2$ and $-\pi/2$ are the *same angle* sitting at opposite ends of the target range, which is
   poison for MSE. The model therefore regresses $(\cos 2\beta,\ \sin 2\beta)$ and decodes with
   $\operatorname{atan2}/2$ — continuous and periodic by construction.

## 5. Generate the training set

Synthesise fresh $h \sim U(-1,1)$ and label each by multi-restart search:

* **explore** — `n_restarts` short Adam ascents per instance, keep the winner;
* **polish** — a longer, lower-LR ascent from that winner only.

Cheaper *and* better than one long multi-restart run, because the expensive fine-tuning is spent
only on the restart that actually won.

**The 500 official `h_train` rows are labelled too but held out of training entirely** — they are
what the main-stage leaderboard scores, so keeping them unseen makes our validation number
directly comparable to it.

The cell calibrates on a pilot batch and **prints a projected runtime before starting the full
run**. Results are cached to `CACHE`; delete it to regenerate.

In [ ]:
def generate_labels(sim, h, n_restarts, steps, polish_steps, lr, rows_per_chunk,
                    seed=0, log=True):
    """Multi-restart angle search for every row of h. Returns canonicalised (gamma, beta, P)."""
    n_h = max(1, rows_per_chunk // n_restarts)
    gen = np.random.default_rng(seed)
    G_out, B_out, V_out = [], [], []
    t0 = time.time()
    for start in range(0, len(h), n_h):
        hc = h[start:start + n_h]
        # ---- explore -----------------------------------------------------------------
        E = sim.energies(np.repeat(hc, n_restarts, axis=0))
        g0, b0 = init_angles(len(hc) * n_restarts, gen)
        g, b, val = adam_ascend(sim, E, sim.ground_mask(E), g0, b0, steps, lr)
        val = val.view(len(hc), n_restarts)
        g = g.view(len(hc), n_restarts, P_DEPTH)
        b = b.view(len(hc), n_restarts, P_DEPTH)
        rows = torch.arange(len(hc), device=val.device)
        pick = val.argmax(dim=1)
        gb, bb = g[rows, pick].contiguous(), b[rows, pick].contiguous()
        # ---- polish the winner -------------------------------------------------------
        E1 = sim.energies(hc)
        gb, bb, vb = adam_ascend(sim, E1, sim.ground_mask(E1), gb, bb, polish_steps, lr * 0.3)
        G_out.append(gb.cpu().numpy()); B_out.append(bb.cpu().numpy()); V_out.append(vb.cpu().numpy())
        if log:
            done = start + len(hc)
            el = time.time() - t0
            print(f"  {done}/{len(h)}  elapsed {el/60:5.1f} min  "
                  f"eta {el/done*(len(h)-done)/60:5.1f} min  meanP={vb.mean():.3f}", end="\r")
    if log:
        print()
    gamma, beta = canonicalize(np.concatenate(G_out), np.concatenate(B_out))
    return gamma, beta, np.concatenate(V_out)


# ---- calibrate on ONE full-size chunk, then project the full cost ------------------------
# Sized to a real chunk, not a small one: GPU throughput depends strongly on batch size, so a
# smaller pilot would mis-estimate the rate.
PILOT = max(32, CFG["rows_per_chunk"] // CFG["n_restarts"])
t0 = time.time()
generate_labels(sim, rng.uniform(-1, 1, (PILOT, N_QUBITS)), CFG["n_restarts"], CFG["steps"],
                CFG["polish_steps"], CFG["lr"], CFG["rows_per_chunk"], log=False)
rate = PILOT / (time.time() - t0)
n_total = CFG["n_new"] + len(h_train)
print(f"pilot: {rate:.1f} instances/s  ->  projected {n_total/rate/60:.1f} min "
      f"for {n_total} instances")
print("(too slow? interrupt, lower CFG['n_new'] / n_restarts / steps, and re-run)")

In [ ]:
if os.path.exists(CACHE):
    d = np.load(CACHE)
    h_all, gamma_all, beta_all = d["h"], d["gamma"], d["beta"]
    p_all, in_pool = d["p_ground"], d["is_train_pool"]
    print(f"loaded cached labels: {len(h_all)} instances")
else:
    h_new = rng.uniform(-1, 1, (CFG["n_new"], N_QUBITS))
    h_all = np.concatenate([h_new, h_train])
    in_pool = np.concatenate([np.ones(len(h_new), bool), np.zeros(len(h_train), bool)])
    t0 = time.time()
    gamma_all, beta_all, _ = generate_labels(
        sim, h_all, CFG["n_restarts"], CFG["steps"], CFG["polish_steps"],
        CFG["lr"], CFG["rows_per_chunk"], seed=SEED)
    p_all = p_ground_ref(h_all, gamma_all, beta_all)   # re-scored with the organisers' simulator
    np.savez_compressed(CACHE, h=h_all, gamma=gamma_all, beta=beta_all,
                        p_ground=p_all, is_train_pool=in_pool, cfg=json.dumps(CFG))
    print(f"generated {len(h_all)} labels in {(time.time()-t0)/60:.1f} min -> {CACHE}")

print(f"\nlabel quality (oracle): mean P = {p_all.mean():.4f} | median {np.median(p_all):.4f} "
      f"| min {p_all.min():.4f} | max {p_all.max():.4f}")
plt.figure(figsize=(6, 3))
plt.hist(p_all, bins=60)
plt.xlabel("P(ground) of the generated label"); plt.ylabel("count")
plt.title("per-instance label quality"); plt.tight_layout(); plt.show()

## 6. Features and targets

**Inputs** — canonicalised `h` plus spectrum descriptors that the rank-1 form makes free to
compute (ground energy, spectral gap, low-energy tail, projections onto $v$). Every block is
invariant under the 128-element symmetry group; the cell asserts it.

**Targets** — $[\gamma_{1..5},\ \cos 2\beta_{1..5},\ \sin 2\beta_{1..5}]$, per §4.

In [ ]:
def canonical_h(h):
    """One representative per symmetry orbit: sort each (i, 11-i) pair descending (kills the 64
    permutations), then pick the lexicographically larger of h / -h (kills the spin flip)."""
    def pair_sort(x):
        x = x.copy()
        for i, j in PAIRS:
            lo, hi = np.minimum(x[:, i], x[:, j]), np.maximum(x[:, i], x[:, j])
            x[:, i], x[:, j] = hi, lo
        return x
    h = np.atleast_2d(h)
    pos, neg = pair_sort(h), pair_sort(-h)
    diff = pos - neg
    first = np.abs(diff) > 1e-12
    idx = np.where(first.any(axis=1), first.argmax(axis=1), 0)
    take_pos = diff[np.arange(len(h)), idx] >= 0
    return np.where(take_pos[:, None], pos, neg)


class FeatureBuilder:
    """h -> model input. Fit the scaler on train only, then reuse for val/test."""

    def __init__(self, J, v, S):
        self.v, self.S = v, S
        self.quad = 0.5 * ((S @ v) ** 2 - (v ** 2).sum())
        self.mean_ = self.std_ = None

    def raw(self, h):
        h = np.atleast_2d(np.asarray(h, float))
        hc = canonical_h(h)
        E = self.quad[None, :] + h @ self.S.T
        order = np.argsort(E, axis=1)
        rows = np.arange(len(h))
        e0 = E[rows, order[:, 0]]
        gap = E[rows, order[:, 1]] - e0
        zg = self.S[order[:, 0]]                       # ground-state spin configuration
        # every block below is constant along a symmetry orbit (E itself is orbit-invariant)
        return np.concatenate([
            hc,                                        # 12  canonical h
            (hc @ self.v)[:, None],                    #  1  projection on v
            np.sort(np.abs(h), axis=1),                # 12  sorted |h|
            np.linalg.norm(h, axis=1, keepdims=True),  #  1
            e0[:, None],                               #  1  ground energy
            gap[:, None],                              #  1  spectral gap
            np.abs(zg @ self.v)[:, None],              #  1  |v.z*|
            np.abs((h * zg).sum(1))[:, None],          #  1  |h.z*|
            E.mean(1)[:, None], E.std(1)[:, None],     #  2
            np.sort(E, axis=1)[:, :8],                 #  8  low-energy tail
        ], axis=1)

    def fit(self, h):
        X = self.raw(h); self.mean_, self.std_ = X.mean(0), X.std(0) + 1e-8; return self

    def transform(self, h):
        return ((self.raw(h) - self.mean_) / self.std_).astype(np.float32)


def encode_angles(gamma, beta):
    """(gamma, beta) -> 15-dim target [gamma, cos 2beta, sin 2beta]."""
    return np.concatenate([gamma, np.cos(2*beta), np.sin(2*beta)], axis=1).astype(np.float32)


def decode_angles(y):
    return y[:, :P_DEPTH], 0.5 * np.arctan2(y[:, 2*P_DEPTH:], y[:, P_DEPTH:2*P_DEPTH])


fb = FeatureBuilder(J, v, S).fit(h_all[in_pool])

X_orb = fb.transform(H_orbit)
print(f"feature spread across a 128-element symmetry orbit: {np.abs(X_orb - X_orb[0]).max():.1e}")
assert np.abs(X_orb - X_orb[0]).max() < 1e-5, "features are not symmetry-invariant"

_g, _b = decode_angles(encode_angles(G, B))
print(f"angle encode/decode round-trip: gamma {np.abs(_g-G).max():.1e}, "
      f"beta {np.abs(wrap_pi(_b-B)).max():.1e}")

X_all = fb.transform(h_all)
Y_all = encode_angles(gamma_all, beta_all)

idx_pool = np.where(in_pool)[0]; rng.shuffle(idx_pool)
n_val = int(0.1 * len(idx_pool))
idx_va, idx_tr = idx_pool[:n_val], idx_pool[n_val:]
idx_holdout = np.where(~in_pool)[0]              # the 500 official h_train instances

print(f"\nfeatures        : {X_all.shape[1]}")
print(f"train           : {len(idx_tr)} synthetic")
print(f"val (synthetic) : {len(idx_va)}")
print(f"holdout         : {len(idx_holdout)} official h_train  <- never seen in training")

## 7. The trivial baseline MLP

3×256 hidden units, GELU, LayerNorm, dropout, AdamW, MSE. Nothing clever — that is the point.
The only non-default choice is the *representation*: the $(\cos,\sin)$ half of the output is
L2-normalised per layer, so the decoded $\beta$ is well-defined from the first epoch.

In [ ]:
class AngleMLP(torch.nn.Module):
    def __init__(self, n_in, hidden=(256, 256, 256), p_drop=0.1):
        super().__init__()
        layers, d = [], n_in
        for hd in hidden:
            layers += [torch.nn.Linear(d, hd), torch.nn.LayerNorm(hd),
                       torch.nn.GELU(), torch.nn.Dropout(p_drop)]
            d = hd
        self.body = torch.nn.Sequential(*layers)
        self.head_gamma = torch.nn.Linear(d, P_DEPTH)
        self.head_beta = torch.nn.Linear(d, 2 * P_DEPTH)

    def forward(self, x):
        z = self.body(x)
        cs = self.head_beta(z).view(-1, 2, P_DEPTH)
        cs = cs / cs.norm(dim=1, keepdim=True).clamp_min(1e-6)   # unit circle per layer
        return torch.cat([self.head_gamma(z), cs[:, 0], cs[:, 1]], dim=1)


def train_mlp(model, Xtr, Ytr, Xva, Yva, epochs=300, batch_size=256, lr=2e-3,
              weight_decay=1e-4, log_every=50):
    model = model.to(DEVICE)
    Xtr_t, Ytr_t = torch.as_tensor(Xtr, device=DEVICE), torch.as_tensor(Ytr, device=DEVICE)
    Xva_t, Yva_t = torch.as_tensor(Xva, device=DEVICE), torch.as_tensor(Yva, device=DEVICE)
    steps_per_epoch = max(1, len(Xtr) // batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
                                                total_steps=epochs * steps_per_epoch)
    hist = []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t), device=DEVICE)
        tot = 0.0
        for i in range(steps_per_epoch):
            idx = perm[i*batch_size:(i+1)*batch_size]
            opt.zero_grad()
            loss = torch.nn.functional.mse_loss(model(Xtr_t[idx]), Ytr_t[idx])
            loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        model.eval()
        with torch.no_grad():
            va = torch.nn.functional.mse_loss(model(Xva_t), Yva_t).item()
        hist.append((ep, tot / steps_per_epoch, va))
        if ep % log_every == 0 or ep == epochs - 1:
            print(f"  epoch {ep:4d}  train {tot/steps_per_epoch:.4f}  val {va:.4f}")
    return model, np.array(hist)


@torch.no_grad()
def predict_angles(model, X, batch=8192):
    model.eval()
    out = [model(torch.as_tensor(X[i:i+batch], device=DEVICE)).cpu().numpy()
           for i in range(0, len(X), batch)]
    return decode_angles(np.concatenate(out))


mlp = AngleMLP(X_all.shape[1])
print(f"parameters: {sum(p.numel() for p in mlp.parameters()):,}\n")
mlp, hist = train_mlp(mlp, X_all[idx_tr], Y_all[idx_tr], X_all[idx_va], Y_all[idx_va])

plt.figure(figsize=(6, 3))
plt.plot(hist[:, 0], hist[:, 1], label="train"); plt.plot(hist[:, 0], hist[:, 2], label="val")
plt.xlabel("epoch"); plt.ylabel("MSE on encoded angles"); plt.legend()
plt.tight_layout(); plt.show()

## 8. Evaluation — the number that counts

Angle MSE is not the metric. We decode the predicted angles and push them through the organisers'
`QAOA.py` on the 500 held-out official instances.

In [ ]:
g_pred, b_pred = predict_angles(mlp, X_all[idx_holdout])
p_mlp = p_ground_ref(h_all[idx_holdout], g_pred, b_pred)
p_oracle = p_all[idx_holdout]

print(f"{'method':<34}{'mean P(ground)':>15}")
print("-" * 49)
for name, val in [("random angles",                 p_random.mean()),
                  ("best CONSTANT angles (banned)", p_const.mean()),
                  ("baseline MLP",                  p_mlp.mean()),
                  ("per-instance oracle (labels)",  p_oracle.mean())]:
    print(f"{name:<34}{val:>15.5f}")
print("-" * 49)
print(f"\nMLP / oracle           : {p_mlp.mean()/p_oracle.mean()*100:.1f}%")
print(f"MLP / constant-angle bar: {p_mlp.mean()/p_const.mean():.2f}x  "
      f"({'PASS - the model uses h' if p_mlp.mean() > p_const.mean() else 'FAIL - no better than a constant'})")
spread = np.concatenate([g_pred, b_pred], 1).std(0)
print(f"prediction std across h : {np.round(spread, 3)}   (all > 0 => output depends on h)")

In [ ]:
# Is the prediction at least a good *starting point*? Polish it with a few local steps.
# This separates "found the right basin" from "found the right point in it".
def polish(h, gamma, beta, steps, lr=0.02):
    E = sim.energies(h)
    g, b, _ = adam_ascend(sim, E, sim.ground_mask(E),
                          torch.tensor(gamma, dtype=torch.float32, device=DEVICE),
                          torch.tensor(beta, dtype=torch.float32, device=DEVICE), steps, lr)
    return canonicalize(g.cpu().numpy(), b.cpu().numpy())


hh = h_all[idx_holdout]
for n_steps in [25, 100]:
    gr, br = polish(hh, g_pred, b_pred, n_steps)
    print(f"MLP      + {n_steps:3d} local steps : P = {p_ground_ref(hh, gr, br).mean():.5f}")
gz, bz = polish(hh, G_CONST[:len(hh)], B_CONST[:len(hh)], 100)
print(f"constant + 100 local steps : P = {p_ground_ref(hh, gz, bz).mean():.5f}"
      "   <- control: how much of the gain is just the polish?")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist([p_mlp, p_oracle], bins=40, label=["baseline MLP", "oracle labels"])
ax[0].axvline(p_const.mean(), c="r", ls="--", label="best constant")
ax[0].set_xlabel("P(ground)"); ax[0].set_ylabel("count"); ax[0].legend(fontsize=8)
ax[0].set_title("held-out official h_train (500 instances)")
ax[1].scatter(p_oracle, p_mlp, s=6, alpha=.4)
lim = max(p_oracle.max(), p_mlp.max()) * 1.05
ax[1].plot([0, lim], [0, lim], "k--", lw=1)
ax[1].set_xlabel("oracle P(ground)"); ax[1].set_ylabel("MLP P(ground)")
ax[1].set_title("per-instance: does the model track the oracle?")
plt.tight_layout(); plt.show()

## 9. Submission file

Required format: 500 rows, columns `id, gamma_0..gamma_4, beta_0..beta_4`. This cell uses
`h_test.npy` as soon as it is present, and falls back to `h_train.npy` so it stays runnable before
the test set is released. Inference must finish inside 10 minutes — it takes well under a second.

In [ ]:
def write_submission(path, model, feature_builder, h):
    g, b = predict_angles(model, feature_builder.transform(h))
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id"] + [f"gamma_{i}" for i in range(P_DEPTH)]
                          + [f"beta_{i}" for i in range(P_DEPTH)])
        for i, (gi, bi) in enumerate(zip(g, b)):
            w.writerow([i] + [f"{x:.8f}" for x in gi] + [f"{x:.8f}" for x in bi])
    return g, b


try:
    h_sub, src = np.load(find_file("h_test.npy")), "h_test.npy"
except FileNotFoundError:
    h_sub, src = h_train, "h_train.npy (h_test not released yet)"

t0 = time.time()
g_sub, b_sub = write_submission("submission.csv", mlp, fb, h_sub)
dt = time.time() - t0
print(f"wrote submission.csv from {src}: {len(h_sub)} rows in {dt:.2f}s (limit 600s)")
print(f"mean P(ground) of the written angles: {p_ground_ref(h_sub, g_sub, b_sub).mean():.5f}")

# save the trained model + scaler so inference can be re-run without retraining
torch.save({"state_dict": mlp.state_dict(), "n_in": X_all.shape[1],
            "feat_mean": fb.mean_, "feat_std": fb.std_}, "baseline_mlp.pt")
print("saved baseline_mlp.pt")

## 10. Where the remaining score is

This baseline is deliberately plain. The diagnostics above point at what to fix, in rough order of
expected payoff:

1. **Train against the simulator, not against labels.** `QAOA.py` is differentiable in the angles,
   so the MLP can be trained to maximise $P(\text{ground})$ end-to-end. That sidesteps the entire
   multi-basin labelling problem of §4 — there is no "correct" angle to average toward, only a
   score to raise. Almost certainly the single biggest win available, and it reuses the labels
   here as a warm start.
2. **Make the labels branch-consistent.** If staying with regression: cluster the canonicalised
   labels, re-optimise each instance seeded from its cluster centroid, and keep the consistent
   solution whenever it costs little $P$. The regression target becomes smooth in $h$.
3. **Predict a distribution, not a point.** A mixture-density head — or $k$ candidate angle vectors
   scored by the simulator at inference — matches the multi-modal structure directly. The 10-minute
   inference budget allows hundreds of simulator calls per instance, which is ample.
4. **Buy better labels.** §4 shows best-of-$k$ still climbing at $k=32$. The oracle is a hard
   ceiling on any model trained against it, so raising it raises everything.
5. **INTERP / layer-wise initialisation.** Grow $p = 1 \to 5$, seeding each depth from the previous
   optimum — the standard QAOA trick for landing in the good basin, and it yields labels that are
   smooth across $h$ by construction.